# 03 — So sánh kết quả đã chạy

Chỉ đọc predictions/manifest/metric; không fit model. Hỗ trợ kết quả notebook01/02 bản cũ và bản mới. Hai lượt đem so phải cùng data, code và giao thức; không trộn kết quả cũ/mới nếu source thay đổi.

## 1. Môi trường và chọn hai thư mục kết quả

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = None  # Ví dụ: r"D:\UIT\DS317\BAF_project_clean"
here = Path.cwd().resolve()
ROOT = (
    Path(PROJECT_DIR).resolve()
    if PROJECT_DIR
    else next(
        (path for path in [here, *here.parents] if (path / "src/baf.py").is_file()),
        None,
    )
)
if ROOT is None:
    raise FileNotFoundError("Mở project BAF_project_clean hoặc điền PROJECT_DIR.")

sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import display
import baf

if Path(baf.__file__).resolve() != (ROOT / "src/baf.py").resolve():
    raise RuntimeError(
        "Kernel đang giữ module của project khác. Restart Kernel trước khi Run All."
    )

print("Project:", ROOT)
print("Kernel:", sys.executable)
print(pd.Series(baf.package_versions()).to_string())


Project: D:\UIT\DS317\BAF_project
Kernel: c:\Users\ADMIN\AppData\Local\Python\pythoncore-3.14-64\python.exe
numpy            2.4.2
pandas           3.0.1
scipy           1.17.1
scikit-learn     1.7.2
psutil           7.2.2
lightgbm         4.7.0


In [2]:
RUNS_ROOT = ROOT / "results/runs"
previous_runs = ROOT.parent / "BAF_project/results/runs"
if not list(RUNS_ROOT.glob("baseline_*/metrics.csv")) and previous_runs.is_dir():
    RUNS_ROOT = previous_runs

BASELINE_DIR = None  # Hoặc Path(r"D:\...\baseline_...")
COST_DIR = None  # Hoặc Path(r"D:\...\cost_...")
REFERENCE = "lgbm_balanced"

for label, value in [("baseline", BASELINE_DIR), ("cost", COST_DIR)]:
    if value is None:
        candidates = sorted(
            path.parent for path in RUNS_ROOT.glob(f"{label}_*/metrics.csv")
        )
        if len(candidates) != 1:
            print("Candidates:", *candidates, sep="\n")
            raise ValueError(
                f"Điền {label.upper()}_DIR để chọn đúng lượt; không tự chọn lượt mới nhất."
            )
        if label == "baseline":
            BASELINE_DIR = candidates[0]
        else:
            COST_DIR = candidates[0]

print("Baseline:", BASELINE_DIR)
print("Cost:", COST_DIR)


Baseline: D:\UIT\DS317\BAF_project\results\runs\baseline_20261027_094947_c40fbc6c
Cost: D:\UIT\DS317\BAF_project\results\runs\cost_20261027_101323_a07d5455


## 2. Kiểm tra và so sánh
Tính lại metric từ predictions; giữ nguyên score dtype, ngưỡng và nhóm row IDs. Các phương pháp có cùng seed sẽ được ghép cặp với reference.

In [3]:
report = baf.audit_results(
    run_paths=[BASELINE_DIR, COST_DIR],
    reference=REFERENCE,
)
display(report.prediction_audit)
display(report.summary)
display(report.deltas)


,method,seed,partition,sha256,verified
0,lr_none,42,validation,3eacf6b6c75e38505216b8f8add1c20405231a46c02b20...,True
1,lr_balanced,42,validation,b342051a9f5eb2e5140f4d54dfb330fbaaaae59f3c156f...,True
2,lgbm_none,42,validation,54881b96717fcdf19b1d9386e321e5ab3d99f7ef6bd205...,True
3,lgbm_none,317,validation,0a726d8abdf59fa95507e5e7ea9b45d1cd1812686f8341...,True
4,lgbm_none,2026,validation,662b9bca3af0ca1153bfeb0e1c0d2d886e41bf6bdb3fee...,True
5,lgbm_balanced,42,validation,e24a5b9a985bdc7bdb59e8835e4c3266df11bc829ee5b1...,True
6,lgbm_balanced,317,validation,6e9f943a27e0843ec14c6c83044befd75ab9c7bc789cc4...,True
7,lgbm_balanced,2026,validation,5bcb4b9c701f64199f9e40a67848c42ee1930fa35cda55...,True
8,lgbm_cost_0.0114,42,validation,18c0ef53c5f76b413b9bd05cc0e69b1f89086a533aad63...,True
9,lgbm_cost_0.0114,317,validation,47cd40ec33c05a923e590e849967e00586b1300c5b4aa1...,True


,method,partition,n_runs,ap_mean,ap_std,recall_mean,recall_std,fit_seconds_mean
0,lgbm_balanced,validation,3,0.181438,0.000743,0.525977,0.002217,28.871141
1,lgbm_cost_0.0114,validation,3,0.181433,0.001620,0.526437,0.003402,29.078262
2,lgbm_cost_0.057,validation,3,0.181358,0.001275,0.525287,0.003185,30.621396
3,lgbm_cost_0.285,validation,3,0.179957,0.000504,0.517011,0.001436,25.370922
4,lgbm_none,validation,3,0.180348,0.001549,0.521839,0.003471,26.696834
5,lr_balanced,validation,1,0.155413,NaN,0.506207,NaN,24.369720
6,lr_none,validation,1,0.155743,NaN,0.513103,NaN,13.338059


,reference,method,seed,partition,delta_recall_pp,delta_ap
0,lgbm_balanced,lr_none,42,validation,-1.034483,-0.026341
1,lgbm_balanced,lr_balanced,42,validation,-1.724138,-0.026670
2,lgbm_balanced,lgbm_none,42,validation,-0.482759,-0.000716
3,lgbm_balanced,lgbm_none,317,validation,-0.137931,-0.002059
4,lgbm_balanced,lgbm_none,2026,validation,-0.620690,-0.000493
5,lgbm_balanced,lgbm_cost_0.0114,42,validation,0.137931,-0.001899
6,lgbm_balanced,lgbm_cost_0.0114,317,validation,0.344828,0.002639
7,lgbm_balanced,lgbm_cost_0.0114,2026,validation,-0.344828,-0.000755
8,lgbm_balanced,lgbm_cost_0.057,42,validation,0.000000,0.000275
9,lgbm_balanced,lgbm_cost_0.057,317,validation,-0.344828,0.001168


## 3. Fraud bắt thêm và bỏ sót
Mỗi model dùng điểm vận hành của nó tại FPR≤5% trên validation. FPR thực tế có thể khác nhau. Đây là phân tích thăm dò, chưa có bootstrap/kiểm định ý nghĩa thống kê.

In [4]:
display(report.overlap)
display(report.changed_cases.head(20))

COMPARISON_DIR = baf.new_directory(ROOT / "results/comparisons", "comparison")
report.save(COMPARISON_DIR)
print("Comparison:", COMPARISON_DIR)


,reference,method,seed,partition,fraud_gained,fraud_lost,net_fraud,baseline_fp,candidate_fp
0,lgbm_balanced,lr_none,42,validation,74,89,-15,5313,5318
1,lgbm_balanced,lr_balanced,42,validation,76,101,-25,5313,5329
2,lgbm_balanced,lgbm_none,42,validation,41,48,-7,5313,5308
3,lgbm_balanced,lgbm_none,317,validation,43,45,-2,5330,5333
4,lgbm_balanced,lgbm_none,2026,validation,35,44,-9,5284,5335
5,lgbm_balanced,lgbm_cost_0.0114,42,validation,23,21,2,5313,5274
6,lgbm_balanced,lgbm_cost_0.0114,317,validation,27,22,5,5330,5287
7,lgbm_balanced,lgbm_cost_0.0114,2026,validation,22,27,-5,5284,5295
8,lgbm_balanced,lgbm_cost_0.057,42,validation,30,30,0,5313,5332
9,lgbm_balanced,lgbm_cost_0.057,317,validation,22,27,-5,5330,5318


,reference,method,seed,partition,row_index,change
0,lgbm_balanced,lr_none,42,validation,795010,gained
1,lgbm_balanced,lr_none,42,validation,796944,lost
2,lgbm_balanced,lr_none,42,validation,797432,gained
3,lgbm_balanced,lr_none,42,validation,797715,gained
4,lgbm_balanced,lr_none,42,validation,797942,gained
5,lgbm_balanced,lr_none,42,validation,798038,lost
6,lgbm_balanced,lr_none,42,validation,798264,lost
7,lgbm_balanced,lr_none,42,validation,798504,lost
8,lgbm_balanced,lr_none,42,validation,798867,lost
9,lgbm_balanced,lr_none,42,validation,799003,lost


Comparison: D:\UIT\DS317\BAF_project\results\comparisons\comparison_20261027_101846_f14113cc


## 4. Xuất proof trong một lần
Một hàm chung như notebook01/02/04. ZIP chứa code kiểm chứng, inputs và output thực thi thật; không cần Ctrl+S trước cell này.

In [5]:
archive = baf.export_proof(
    run_paths=[BASELINE_DIR, COST_DIR],
    output_root=ROOT / "results/proofs",
    reference=REFERENCE,
)


Verified prediction files: 17
Proof ZIP: D:\UIT\DS317\BAF_project\results\proofs\proof_20261027_101846_b7f360c0.zip
